# Downstream Evaluation of Nicheformer Embeddings

Following the methodology from **Nicheformer: a foundation model for single-cell and spatial omics** (Nature Methods 2025, Vol. 22, pp. 2525-2538)

## Tasks (from the paper):

### 1. Spatial Label Prediction (Classification → macro F1)
- **Cell-type classification**: Predict `author_cell_type` from frozen embeddings
- **Niche classification**: Predict `niche` labels from frozen embeddings
- **Region classification**: Predict `region` labels from frozen embeddings

### 2. Spatial Composition Prediction (Regression → Mean Absolute Error)
- **Niche composition regression**: Predict neighborhood cell-type composition vectors (`X_niche_0`..`X_niche_4`) at multiple radii
- **Neighborhood density regression**: Predict local cell density

### 3. Transfer Learning Settings
- **Linear probing**: Frozen embeddings + linear classifier/regressor (LogisticRegression / Ridge)
- **Fine-tuning**: Full model fine-tuning using `NicheformerFineTune` with PyTorch Lightning

### Data
- CosMx human liver Cancerous dataset
- 368,226 cells × 512-dim Nicheformer embeddings
- 17 cell types, 8 niches, 1 region
- Neighborhood composition at 5 radii (X_niche_0..X_niche_4)

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import issparse, csr_matrix
from sklearn.linear_model import LogisticRegression, Ridge, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, mean_absolute_error, r2_score, mean_squared_error
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [2]:
# Load the data with embeddings
DATA_PATH = "/mnt/172/wh/25-12/spatial/adata_with_embeddings.h5ad"
emb = sc.read_h5ad(DATA_PATH)
print(f"Loaded AnnData: {emb.n_obs} cells \u00d7 {emb.n_vars} genes")
print(f"Embeddings shape: {emb.obsm['X_nicheformer_embeddings'].shape}")
print(f"\nObs columns: {list(emb.obs.columns)}")
print(f"ObSm keys: {list(emb.obsm.keys())}")
print(f"\nUnique values:")
for col in ['author_cell_type', 'niche', 'region', 'batch']:
    n_unique = emb.obs[col].nunique()
    vals = sorted(emb.obs[col].unique())
    print(f"  {col}: {n_unique} unique values \u2192 {vals[:10]}{'...' if len(vals) > 10 else ''}")

Loaded AnnData: 368226 cells × 20310 genes
Embeddings shape: (368226, 512)

Obs columns: ['assay', 'organism', 'nicheformer_split', 'batch', 'niche', 'region', 'author_cell_type', 'modality', 'specie']
ObSm keys: ['X_niche_0', 'X_niche_1', 'X_niche_2', 'X_niche_3', 'X_niche_4', 'X_nicheformer_embeddings', 'X_pca', 'X_umap', 'X_umap_bySlide', 'falsecode_counts', 'negprobes_counts', 'spatial', 'spatial_fov_px']

Unique values:
  author_cell_type: 17 unique values → [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]...
  niche: 4 unique values → [0, 1, 2, 3]
  region: 1 unique values → [0]
  batch: 1 unique values → ['2']


---
## 1. Spatial Label Prediction (Linear Probing)

Following the paper: train a **linear classifier** (LogisticRegression) on **frozen Nicheformer embeddings**.
Report **macro F1** as the primary metric (matching the paper's evaluation).

In [ ]:
def evaluate_linear_probing_classification(emb, label_col, task_name, test_size=0.2, random_state=42):
    """
    Linear probing: train LogisticRegression on frozen embeddings.
    Reports macro F1, weighted F1, accuracy, and per-class F1.
    """
    X = emb.obsm['X_nicheformer_embeddings']
    y = emb.obs[label_col].values
    
    # Encode labels if they're not already integers
    if y.dtype not in [np.int32, np.int64, np.float32, np.float64]:
        le = LabelEncoder()
        y = le.fit_transform(y)
    
    # Stratified split to preserve class distribution
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    # Standardize embeddings (important for linear models)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Linear probing: LogisticRegression with high max_iter for convergence
    clf = LogisticRegression(
        multi_class='multinomial',
        solver='lbfgs',
        max_iter=5000,
        random_state=random_state,
        n_jobs=-1
    )
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    weighted_f1 = f1_score(y_test, y_pred, average='weighted')
    
    print(f"\n{'='*60}")
    print(f"Task: {task_name} (Linear Probing)")
    print(f"{'='*60}")
    print(f"  Classes: {len(np.unique(y))}")
    print(f"  Train size: {len(y_train)}, Test size: {len(y_test)}")
    print(f"  Accuracy:    {acc:.4f}")
    print(f"  Macro F1:    {macro_f1:.4f}")
    print(f"  Weighted F1: {weighted_f1:.4f}")
    
    # Per-class F1
    per_class_f1 = f1_score(y_test, y_pred, average=None)
    print(f"\n  Per-class F1 (top 15 classes):")
    class_labels = np.unique(y)
    for i in np.argsort(-per_class_f1)[:15]:
        support = np.sum(y_test == class_labels[i])
        print(f"    Class {class_labels[i]}: F1={per_class_f1[i]:.4f} (n={support})")
    
    return {
        'task': task_name,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'y_test': y_test,
        'y_pred': y_pred
    }


results_classification = {}

# Cell-type classification
results_classification['cell_type'] = evaluate_linear_probing_classification(
    emb, 'author_cell_type', 'Cell-Type Classification'
)

# Niche classification
results_classification['niche'] = evaluate_linear_probing_classification(
    emb, 'niche', 'Niche Classification'
)


### 1b. Comparison: Random Forest Classifier (Non-linear Baseline)

The paper also compares against non-linear methods. RandomForest provides a strong non-linear baseline.

In [ ]:
def evaluate_rf_classification(emb, label_col, task_name, test_size=0.2, random_state=42):
    X = emb.obsm['X_nicheformer_embeddings']
    y = emb.obs[label_col].values
    
    if y.dtype not in [np.int32, np.int64, np.float32, np.float64]:
        le = LabelEncoder()
        y = le.fit_transform(y)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=random_state)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    weighted_f1 = f1_score(y_test, y_pred, average='weighted')
    
    print(f"\n{'='*60}")
    print(f"Task: {task_name} (Random Forest)")
    print(f"{'='*60}")
    print(f"  Accuracy:    {acc:.4f}")
    print(f"  Macro F1:    {macro_f1:.4f}")
    print(f"  Weighted F1: {weighted_f1:.4f}")
    
    return {'task': task_name, 'accuracy': acc, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1}


print("\n" + "="*60)
print("NON-LINEAR BASELINE: Random Forest")
print("="*60)

rf_results = {}
rf_results['cell_type'] = evaluate_rf_classification(emb, 'author_cell_type', 'Cell-Type Classification')
rf_results['niche'] = evaluate_rf_classification(emb, 'niche', 'Niche Classification')
rf_results['region'] = evaluate_rf_classification(emb, 'region', 'Region Classification')

### 1c. Summary Table: Classification Results

In [ ]:
summary_rows = []
for task_name in ['cell_type', 'niche', 'region']:
    lr = results_classification[task_name]
    rf = rf_results[task_name]
    summary_rows.append({
        'Task': task_name.replace('_', ' ').title(),
        'Method': 'Linear Probing (LogReg)',
        'Accuracy': f"{lr['accuracy']:.4f}",
        'Macro F1': f"{lr['macro_f1']:.4f}",
        'Weighted F1': f"{lr['weighted_f1']:.4f}"
    })
    summary_rows.append({
        'Task': task_name.replace('_', ' ').title(),
        'Method': 'Random Forest',
        'Accuracy': f"{rf['accuracy']:.4f}",
        'Macro F1': f"{rf['macro_f1']:.4f}",
        'Weighted F1': f"{rf['weighted_f1']:.4f}"
    })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "="*70)
print("CLASSIFICATION RESULTS SUMMARY")
print("="*70)
display(summary_df)

# Bar plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, task_name in enumerate(['cell_type', 'niche', 'region']):
    lr = results_classification[task_name]
    rf = rf_results[task_name]
    axes[i].bar(['Linear Probing', 'Random Forest'], [lr['macro_f1'], rf['macro_f1']], 
                color=['#4C72B0', '#DD8452'], alpha=0.8)
    axes[i].set_title(task_name.replace('_', ' ').title())
    axes[i].set_ylabel('Macro F1')
    axes[i].set_ylim([0, 1])
    for j, v in enumerate([lr['macro_f1'], rf['macro_f1']]):
        axes[i].text(j, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
plt.suptitle('Spatial Label Prediction: Macro F1 Score', fontsize=14)
plt.tight_layout()
plt.show()

---
## 2. Spatial Composition Prediction (Regression → Mean Absolute Error)

Following the paper: predict **neighborhood composition vectors** from frozen embeddings.
The paper defines neighborhoods at multiple radii (mean 10, 20, 50, 100 neighbors).
Here `X_niche_0`..`X_niche_4` represent composition at 5 different radii.

**Metric**: Mean Absolute Error (MAE) — matching the paper.

In [ ]:
def evaluate_niche_composition_regression(emb, niche_keys, test_size=0.2, random_state=42):
    """
    Linear probing for niche composition regression.
    Predicts the neighborhood composition vector at each radius.
    """
    X = emb.obsm['X_nicheformer_embeddings']
    
    # Standardize embeddings
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    results = []
    
    for niche_key in niche_keys:
        y = emb.obsm[niche_key]
        
        # Convert sparse to dense if needed
        if issparse(y):
            y = y.toarray()
        
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=test_size, random_state=random_state
        )
        
        # Ridge regression (linear probing)
        model = Ridge(alpha=1.0)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Metrics
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        # Per-component MAE
        per_component_mae = np.mean(np.abs(y_test - y_pred), axis=0)
        
        results.append({
            'niche_key': niche_key,
            'n_components': y.shape[1],
            'MAE': mae,
            'RMSE': rmse,
            'R\u00b2': r2,
            'per_component_mae_mean': np.mean(per_component_mae),
            'per_component_mae_std': np.std(per_component_mae),
        })
        
        print(f"  {niche_key:12s} ({y.shape[1]:2d} components) \u2192 MAE={mae:.4f}, RMSE={rmse:.4f}, R\u00b2={r2:.4f}")
    
    return results


niche_keys = ['X_niche_0', 'X_niche_1', 'X_niche_2', 'X_niche_3', 'X_niche_4']

print("="*70)
print("SPATIAL COMPOSITION PREDICTION (Linear Probing - Ridge Regression)")
print("="*70)

niche_reg_results = evaluate_niche_composition_regression(emb, niche_keys)

# Summary DataFrame
niche_reg_df = pd.DataFrame(niche_reg_results)
print("\nSummary:")
display(niche_reg_df[['niche_key', 'n_components', 'MAE', 'RMSE', 'R\u00b2']])

### 2b. Neighborhood Density Regression

Predict the total cell density in each neighborhood (sum of composition vector).

In [ ]:
def evaluate_density_regression(emb, niche_keys, test_size=0.2, random_state=42):
    """
    Predict neighborhood density (sum of composition) from embeddings.
    """
    X = emb.obsm['X_nicheformer_embeddings']
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    results = []
    
    for niche_key in niche_keys:
        y = emb.obsm[niche_key]
        if issparse(y):
            y = y.toarray()
        
        # Density = sum of composition (total cells in neighborhood)
        density = np.array(y.sum(axis=1)).flatten()
        
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, density, test_size=test_size, random_state=random_state
        )
        
        model = Ridge(alpha=1.0)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        results.append({
            'niche_key': niche_key,
            'MAE': mae,
            'RMSE': rmse,
            'R\u00b2': r2,
            'mean_density': np.mean(density),
            'std_density': np.std(density)
        })
        
        print(f"  {niche_key:12s} \u2192 Density MAE={mae:.4f}, RMSE={rmse:.4f}, R\u00b2={r2:.4f} (mean density={np.mean(density):.2f})")
    
    return results


print("\n" + "="*70)
print("NEIGHBORHOOD DENSITY PREDICTION (Linear Probing - Ridge Regression)")
print("="*70)

density_results = evaluate_density_regression(emb, niche_keys)

density_df = pd.DataFrame(density_results)
print("\nSummary:")
display(density_df[['niche_key', 'MAE', 'RMSE', 'R\u00b2', 'mean_density', 'std_density']])

### 2c. Visualization: Composition Prediction per Component

In [ ]:
# Visualize per-component MAE for each niche radius
fig, axes = plt.subplots(1, len(niche_keys), figsize=(5*len(niche_keys), 4))

X = emb.obsm['X_nicheformer_embeddings']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

for idx, niche_key in enumerate(niche_keys):
    y = emb.obsm[niche_key]
    if issparse(y):
        y = y.toarray()
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42
    )
    
    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    per_component_mae = np.mean(np.abs(y_test - y_pred), axis=0)
    
    axes[idx].bar(range(len(per_component_mae)), per_component_mae, color='steelblue', alpha=0.7)
    axes[idx].set_title(f'{niche_key}\n({y.shape[1]} components)')
    axes[idx].set_xlabel('Component')
    axes[idx].set_ylabel('MAE')
    axes[idx].axhline(y=np.mean(per_component_mae), color='red', linestyle='--', 
                      label=f'Mean MAE={np.mean(per_component_mae):.4f}')
    axes[idx].legend(fontsize=8)

plt.suptitle('Per-Component MAE for Niche Composition Prediction', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Fine-Tuning (Full Model Training)

Following the paper's **fine-tuning** setting: update the transformer backbone parameters
along with the linear head. This uses the `NicheformerFineTune` model with PyTorch Lightning.

**Note**: Since all data is labeled as 'train' in `nicheformer_split`, we need to create
our own train/val/test split for proper evaluation.

In [ ]:
import os
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from torch.utils.data import DataLoader
import anndata as ad

from nicheformer.models._nicheformer import Nicheformer
from nicheformer.models._nicheformer_fine_tune import NicheformerFineTune
from nicheformer.data.dataset import NicheformerDataset

# Check CUDA availability
device = 'gpu' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Configuration for fine-tuning
CHECKPOINT_PATH = "/mnt/172/wh/25-12/nicheformer/ckpt/nicheformer.ckpt"
TECH_MEAN_PATH = "/mnt/172/wh/25-12/nicheformer/data/model_means/cosmx_mean_script.npy"
OUTPUT_DIR = "/mnt/172/wh/25-12/spatial/checkpoints/nicheformer_ft_logs"

config_ft = {
    'batch_size': 16,
    'max_seq_len': 1500,
    'aux_tokens': 30,
    'chunk_size': 1000,
    'num_workers': 4,
    'precision': 32,
    'max_epochs': 50,
    'lr': 1e-4,
    'warmup': 10,
    'gradient_clip_val': 1.0,
    'accumulate_grad_batches': 10,
    
    # Model parameters
    'extract_layers': [11],
    'function_layers': 'mean',
    'freeze': False,  # Fine-tuning: update backbone
    'reinit_layers': None,
    'extractor': False,
    'regress_distribution': True,
    'pool': 'mean',
    'predict_density': False,
    'ignore_zeros': False,
    'organ': 'liver',
    'without_context': True,
}

print("Configuration loaded.")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Tech mean: {TECH_MEAN_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# Create train/val/test splits from the data
# Since all data is 'train', we create random splits
np.random.seed(42)

# Load the AnnData
adata_ft = ad.read_h5ad(DATA_PATH)
n_cells = adata_ft.n_obs

# Create random indices for splits
indices = np.random.permutation(n_cells)
n_train = int(n_cells * 0.7)
n_val = int(n_cells * 0.15)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train+n_val]
test_idx = indices[n_train+n_val:]

# Assign splits
adata_ft.obs['nicheformer_split'] = 'train'  # reset all
adata_ft.obs.iloc[train_idx, adata_ft.obs.columns.get_loc('nicheformer_split')] = 'train'
adata_ft.obs.iloc[val_idx, adata_ft.obs.columns.get_loc('nicheformer_split')] = 'val'
adata_ft.obs.iloc[test_idx, adata_ft.obs.columns.get_loc('nicheformer_split')] = 'test'

print(f"Split sizes: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")
print(f"Split distribution:\n{adata_ft.obs['nicheformer_split'].value_counts()}")

In [ ]:
# Load technology mean
technology_mean = np.load(TECH_MEAN_PATH)
print(f"Technology mean shape: {technology_mean.shape}")

# Create datasets for niche classification
print("\nCreating datasets for niche classification...")

train_dataset = NicheformerDataset(
    adata=adata_ft,
    technology_mean=technology_mean,
    split='train',
    max_seq_len=config_ft['max_seq_len'],
    aux_tokens=config_ft['aux_tokens'],
    chunk_size=config_ft['chunk_size'],
    metadata_fields={
        'obs': ['niche', 'author_cell_type', 'region', 'modality', 'assay', 'specie'],
    }
)

val_dataset = NicheformerDataset(
    adata=adata_ft,
    technology_mean=technology_mean,
    split='val',
    max_seq_len=config_ft['max_seq_len'],
    aux_tokens=config_ft['aux_tokens'],
    chunk_size=config_ft['chunk_size'],
    metadata_fields={
        'obs': ['niche', 'author_cell_type', 'region', 'modality', 'assay', 'specie'],
    }
)

test_dataset = NicheformerDataset(
    adata=adata_ft,
    technology_mean=technology_mean,
    split='test',
    max_seq_len=config_ft['max_seq_len'],
    aux_tokens=config_ft['aux_tokens'],
    chunk_size=config_ft['chunk_size'],
    metadata_fields={
        'obs': ['niche', 'author_cell_type', 'region', 'modality', 'assay', 'specie'],
    }
)

print(f"\nDataset sizes: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config_ft['batch_size'],
    shuffle=True,
    num_workers=config_ft['num_workers'],
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config_ft['batch_size'],
    shuffle=False,
    num_workers=config_ft['num_workers'],
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config_ft['batch_size'],
    shuffle=False,
    num_workers=config_ft['num_workers'],
    pin_memory=True
)

print("Dataloaders created successfully.")

In [ ]:
# Load pre-trained model
print("Loading pre-trained Nicheformer checkpoint...")
model = Nicheformer.load_from_checkpoint(checkpoint_path=CHECKPOINT_PATH, strict=False)
print("Model loaded successfully.")

# Create fine-tuning model for niche classification
fine_tune_model = NicheformerFineTune(
    backbone=model,
    supervised_task='niche_classification',
    extract_layers=config_ft['extract_layers'],
    function_layers=config_ft['function_layers'],
    lr=config_ft['lr'],
    warmup=config_ft['warmup'],
    max_epochs=config_ft['max_epochs'],
    dim_prediction=1,
    n_classes=emb.obs['niche'].nunique(),
    freeze=config_ft['freeze'],
    reinit_layers=config_ft['reinit_layers'],
    extractor=config_ft['extractor'],
    regress_distribution=False,
    pool=config_ft['pool'],
    predict_density=config_ft['predict_density'],
    ignore_zeros=config_ft['ignore_zeros'],
    organ=config_ft['organ'],
    label='niche',
    without_context=config_ft['without_context']
)

print("Fine-tuning model created.")
print(f"  Task: niche_classification")
print(f"  Classes: {emb.obs['niche'].nunique()}")
print(f"  Freeze backbone: {config_ft['freeze']}")

In [ ]:
# Configure trainer
checkpoint_callback = ModelCheckpoint(
    dirpath=OUTPUT_DIR,
    filename='nicheformer-ft-niche-{epoch:02d}-{val_classification_loss:.4f}',
    monitor='val/classification_loss',
    mode='min',
    save_top_k=3
)

early_stop_callback = EarlyStopping(
    monitor='val/classification_loss',
    patience=10,
    mode='min'
)

trainer = pl.Trainer(
    max_epochs=config_ft['max_epochs'],
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    default_root_dir=OUTPUT_DIR,
    precision=config_ft['precision'],
    gradient_clip_val=config_ft['gradient_clip_val'],
    accumulate_grad_batches=config_ft['accumulate_grad_batches'],
    callbacks=[checkpoint_callback, early_stop_callback],
    enable_progress_bar=True
)

print("Trainer configured.")

In [ ]:
# Train the model
print("Starting fine-tuning training...")
print(f"  Epochs: {config_ft['max_epochs']}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")

trainer.fit(
    model=fine_tune_model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader
)

print("\nFine-tuning complete!")

In [ ]:
# Test the model
print("Testing the fine-tuned model...")
test_results = trainer.test(
    model=fine_tune_model,
    dataloaders=test_loader
)

print(f"\nTest results: {test_results}")

In [ ]:
# Get predictions on test set
print("Getting predictions...")
predictions = trainer.predict(fine_tune_model, dataloaders=test_loader)

# Concatenate predictions
all_preds = torch.cat([p[0] for p in predictions]).cpu().numpy()
all_logits = torch.cat([p[1] for p in predictions]).cpu().numpy()

print(f"Predictions shape: {all_preds.shape}")
print(f"Logits shape: {all_logits.shape}")

# Get ground truth labels
test_indices = adata_ft.obs['nicheformer_split'] == 'test'
true_labels = adata_ft.obs.loc[test_indices, 'niche'].values.astype(int)

# Compute metrics
test_acc = accuracy_score(true_labels, all_preds)
test_macro_f1 = f1_score(true_labels, all_preds, average='macro')

print(f"\nTest Set Results (Fine-Tuned Niche Classification):")
print(f"  Accuracy:  {test_acc:.4f}")
print(f"  Macro F1:  {test_macro_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(true_labels, all_preds))

---
## 4. Summary: Linear Probing vs Fine-Tuning

Compare the results from both transfer learning settings.

In [ ]:
# Build comparison table
comparison_rows = []

# Linear probing results (from Section 1)
for task_name, display_name in [('cell_type', 'Cell-Type'), ('niche', 'Niche'), ('region', 'Region')]:
    lr = results_classification[task_name]
    comparison_rows.append({
        'Task': display_name,
        'Setting': 'Linear Probing',
        'Method': 'LogisticRegression',
        'Macro F1': f"{lr['macro_f1']:.4f}",
        'Accuracy': f"{lr['accuracy']:.4f}"
    })
    rf = rf_results[task_name]
    comparison_rows.append({
        'Task': display_name,
        'Setting': 'Linear Probing',
        'Method': 'RandomForest',
        'Macro F1': f"{rf['macro_f1']:.4f}",
        'Accuracy': f"{rf['accuracy']:.4f}"
    })

# Fine-tuning results (from Section 3) - only niche classification
try:
    comparison_rows.append({
        'Task': 'Niche',
        'Setting': 'Fine-Tuning',
        'Method': 'NicheformerFineTune',
        'Macro F1': f"{test_macro_f1:.4f}",
        'Accuracy': f"{test_acc:.4f}"
    })
except NameError:
    pass

comparison_df = pd.DataFrame(comparison_rows)
print("="*70)
print("COMPARISON: LINEAR PROBING vs FINE-TUNING")
print("="*70)
display(comparison_df)

In [ ]:
# Save evaluation results to CSV
output_dir = "/mnt/172/wh/25-12/spatial/evaluation_results"
os.makedirs(output_dir, exist_ok=True)

# Save classification summary
summary_df.to_csv(os.path.join(output_dir, "classification_results.csv"), index=False)

# Save niche regression summary
niche_reg_df.to_csv(os.path.join(output_dir, "niche_composition_regression.csv"), index=False)

# Save density regression summary
density_df.to_csv(os.path.join(output_dir, "density_regression.csv"), index=False)

# Save comparison
comparison_df.to_csv(os.path.join(output_dir, "linear_vs_finetune_comparison.csv"), index=False)

print(f"All results saved to: {output_dir}")
print("\nFiles created:")
for f in os.listdir(output_dir):
    print(f"  {f}")